# PetCut · Segmentación semántica · AAVA 2026–2027
Dos sesiones de 105 minutos y dos semanas. Desarrollo en **Linux local**, entrenamiento/evaluación en **Colab Free**, demo obligatoria **local CPU**. Lee ENUNCIADO.md y CONTRATOS.md. Completa T0–T7 en los .py y guarda antes de subir.

Este notebook es el registro de evidencias. En Colab abre el notebook y ejecuta preparación/subida del proyecto completo (no solo el ipynb). En una sesión nueva sube la misma versión para continuar. Reinicia el kernel si importaste módulos antes de editarlos. Las pruebas y scripts se ejecutan en procesos nuevos.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    import google.colab
    COLAB = True
except ImportError:
    COLAB = False
PROJECT = Path('/content/P2_SemanticSegmentation') if COLAB else Path.cwd()
if COLAB and not (PROJECT / 'src').is_dir():
    from google.colab import files
    uploaded = files.upload()  # Selecciona PetCut_trabajo.zip creado en Linux
    archives = [name for name in uploaded if name.endswith('.zip')]
    assert len(archives) == 1, 'Sube un único ZIP con raíz P2_SemanticSegmentation/'
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.namelist():
            target = (Path('/content') / member).resolve()
            assert target.is_relative_to(Path('/content')), 'Ruta ZIP inválida'
        archive.extractall('/content')
assert (PROJECT / 'src').is_dir(), 'Ajusta PROJECT a la carpeta que contiene src/'
os.chdir(PROJECT)
print('Proyecto:', PROJECT, 'Colab:', COLAB)


## Entorno
Colab conserva su pareja torch/torchvision: instala solo requisitos adicionales. En Linux instala primero el entorno CPU del README. Si cambias torch después de importarlo, reinicia. Si torchvision falla, no entrenes: recupera una pareja compatible o un runtime nuevo. No se garantiza disponibilidad de GPU.


In [ ]:
if COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'])
import torch, torchvision
from torchvision.models import resnet18
probe = resnet18(weights=None)  # Comprueba la API y la pareja instalada, sin red
del probe


In [ ]:
def run(*args):
    command = [sys.executable, '-u', *map(str, args)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1,
                          env={**os.environ, 'PYTHONIOENCODING': 'utf-8'}) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
print('Python:', sys.version.split()[0], 'torch:', torch.__version__, 'vision:', torchvision.__version__)
print('CUDA disponible:', torch.cuda.is_available())


## Configuración y persistencia: revisa ANTES del primer entrenamiento
**EXPERIMENT** identifica un experimento independiente. En Colab se monta Drive y resultados/manifiestos viven allí; datos se extraen en /content para evitar muchas lecturas en Drive. En sesiones posteriores mantén EXACTAMENTE experiment, perfil, regla y código. Guarda también tu ZIP fuente en Drive.

**MODE='entrenar'** inicia o reanuda automáticamente desde last.pt. **MODE='verificar'** reutiliza resultados y no vuelve a entrenar: úsalo para ejecutar en orden desde kernel limpio antes de entregar.

20 épocas son un presupuesto TOTAL sugerido, ampliable; cinco son orientativas. Implementa tu regla en T7 y configura stopping.json antes de entrenar. CPU_SMALL cambia tamaños/batch, no obliga a una parada concreta. Más imágenes están permitidas si las justificas: usa nuevo manifest/EXPERIMENT; al menos 30 val y 4 test.


In [ ]:
EXPERIMENT = 'petcut-01'
MODE = 'entrenar' if COLAB else 'verificar'  # Cambia explícitamente según tu objetivo
CPU_SMALL = False
COUNTS = (120, 30, 30) if CPU_SMALL else (1500, 300, 300)
BATCH = 8 if CPU_SMALL else 16
EPOCH_BUDGET = 20
SIZE, SEED, LR = 128, 2026, .001
assert MODE in ('entrenar', 'verificar')
assert COUNTS[1] >= 30 and COUNTS[2] >= 4
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = Path('/content/drive/MyDrive/PetCut') / EXPERIMENT
else:
    PERSIST = PROJECT
RESULTS = PERSIST / 'results'
MANIFEST = PERSIST / 'splits' / ('cpu.json' if CPU_SMALL else 'default.json')
SUBSET = PERSIST / 'splits' / ('cpu-inference.json' if CPU_SMALL else 'inference.json')
DATA = PROJECT / 'data'
PERSIST.mkdir(parents=True, exist_ok=True)
print('Resultados persistentes:', RESULTS)


## 1. T0 · Datos y transformaciones (10 %)
Completa los seis huecos T0 de dataset.py localmente. Descarga y selección están resueltas. La descarga sigue siendo completa en ambos perfiles y necesita varios GB; puede ser lenta. Una copia comprimida en Drive evita repetirla, pero las imágenes deben extraerse localmente en el runtime.


In [ ]:
run('-m', 'pytest', '-q', '-k', 'data or trimap or joint_flip or normalization or original or subset or manifest')
run('prepare_data.py', '--data', DATA, '--download', '--output', MANIFEST,
    '--counts', *COUNTS, '--seed', SEED, '--inference-subset', SUBSET)


In [ ]:
import matplotlib.pyplot as plt
import json
from src.dataset import PetDataset, MEAN, STD
from src.visualization import mask_rgb
from IPython.display import display, Image as DisplayImage
ds = PetDataset(DATA, MANIFEST, 'train', size=SIZE, augment=False)
fig, axes = plt.subplots(4, 2, figsize=(7, 12))
for i in range(4):
    x, y = ds[i]
    rgb = (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).permute(1, 2, 0).clamp(0, 1)
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title(ds.ids[i])
    axes[i, 1].imshow(mask_rgb(y))
    axes[i, 1].set_title(f'Etiquetas: {torch.unique(y).tolist()}')
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()
plt.show()


**Responde:** significado de 0/1/255; nearest frente a bilinear; flip conjunto; normalización; qué pierde 128×128; cómo compruebas alineación y ausencia de fuga. Justifica tamaños elegidos.

_Respuesta:_


## 2. T1 y T4 · FCN-lite y métricas (10 % / 15 %)
Completa decoder.py y metrics.py según los contratos. Logits B×2×H×W; encoder congelado incluso en BatchNorm. Confusión global, sin 255. Calcula TP/FP/FN e IoU para real=[0,0,1,1,255], pred=[0,1,1,0,1]. ¿Por qué accuracy no basta?

_Respuesta y esquema:_


In [ ]:
run('-m', 'pytest', '-q', '-k', 'fcn_logits_shape or metrics')


## 3. T2–T3 · Decoder con skips (20 %)
Completa DecoderBlock y SkipSegmenter. Dibuja canales y tamaños, incluida una entrada impar. Explica contexto frente a localización y por qué comparar FCN/skips no aísla capacidad de conexiones.

_Respuesta y esquema:_


In [ ]:
run('-m', 'pytest', '-q', '-k', 'decoder or models_update_only_decoder')


## 4. T7 · Regla de parada propia (15 % con entrenamiento)
Implementa should_stop en stopping.py y configura stopping.json ANTES de entrenar. Contrato determinista `(bool,motivo)`, sin test ni nombre del modelo. Añade dos tests propios (continuación y parada) en tests/test_stopping_custom.py. Las pruebas comunes solo verifican contrato/pureza, no te imponen paciencia concreta. Usa la misma regla/settings para ambos modelos; épocas finales pueden diferir.

**Escribe antes de entrenar:** hipótesis, regla, parámetros, justificación, dos historiales de prueba y presupuesto total. No cambies esta respuesta retrospectivamente sin registrar otro experimento.

_Respuesta:_


In [ ]:
run('-m', 'pytest', '-q', '-k', 'stopping')
print(Path('stopping.json').read_text(encoding='utf-8'))


## 5. Entrenar, guardar y CONTINUAR tras un corte
`last.pt` conserva modelo, optimizador, historial, mejor modelo y estados aleatorios desde la última época COMPLETA. `best.pt` es para evaluar y demo. No recuperes desde best.pt. Una época cortada se repite completa; antes de la primera no existe last.pt y tendrás que elegir otra carpeta para reiniciar.

**Sesión nueva:** subir el MISMO código → instalar → montar Drive → mismo EXPERIMENT/configuración → preparar datos → ejecutar estas celdas en modo entrenar. Detectan last.pt y añaden --resume. Si un modelo ya terminó, no vuelve a entrenarse salvo que amplíes presupuesto y su regla permita seguir. `--epochs` es TOTAL, no épocas extra. Si cambias código/regla/versiones/protocolo inicia otro experimento; un cambio de dispositivo puede alterar resultados.

Modo verificar omite estas ejecuciones y comprueba archivos existentes. No requiere recomputar entrenamientos para ejecutar el notebook desde un kernel limpio.


In [ ]:
common = ['--data', DATA, '--manifest', MANIFEST, '--epochs', EPOCH_BUDGET,
          '--batch-size', BATCH, '--size', SIZE, '--seed', SEED, '--lr', LR,
          '--stopping', 'stopping.json']
def train_or_verify(name):
    folder = RESULTS / name
    if MODE == 'entrenar':
        resume = ['--resume'] if (folder / 'last.pt').exists() else []
        run('train.py', '--model', name, '--out', folder, *common, *resume)
    else:
        required = ['best.pt', 'config.json', 'history.json', 'summary.json', 'curves.png']
        assert all((folder / f).is_file() for f in required), f'Faltan resultados de {name}; entrena o recupera Drive'
train_or_verify('fcn')


In [ ]:
train_or_verify('skip')


In [ ]:
for name in ('fcn', 'skip'):
    config = json.loads((RESULTS / name / 'config.json').read_text(encoding='utf-8'))
    summary = json.loads((RESULTS / name / 'summary.json').read_text(encoding='utf-8'))
    print(name, 'parámetros:', config['trainable_parameters'], 'resumen:', summary)
    display(DisplayImage(filename=str(RESULTS / name / 'curves.png')))


**Analiza:** curvas, parámetros actualizados, tiempo, motivo de parada y época elegida; criterio frente a límite de recursos; interrupciones/reanudación; límites del perfil reducido. ¿Justifican los datos la hipótesis previa? ¿Qué no puedes concluir?

_Respuesta:_


## 6. T5 · Inferencia obligatoria: cuatro combinaciones (20 % con análisis)
Completa predict_mask (resize logits→argmax) y predict_mask_labels (argmax→nearest). Usa los dos best.pt y las 30 val fijas, GT original y 255 excluido. Esta tabla es de VALIDACIÓN ORIGINAL; no se mezcla con la tabla final de test a resolución de entrada. No vuelvas a ajustar entrenamiento para favorecerla.


In [ ]:
run('compare_inference.py', '--data', DATA, '--manifest', MANIFEST, '--subset', SUBSET,
    '--fcn', RESULTS / 'fcn/best.pt', '--skip', RESULTS / 'skip/best.pt', '--out', RESULTS / 'inference')
import pandas as pd
display(pd.read_csv(RESULTS / 'inference/comparison.csv'))
display(DisplayImage(filename=str(RESULTS / 'inference/examples.png')))


**Explica:** diferencias visuales y métricas A/B en cada arquitectura, fracción de píxeles válidos distintos, coste y límites de tiempos. GT permanece original. La figura automática es exploratoria: identifica ejemplos concretos del subconjunto. Diferencias nulas son resultados válidos.

_Respuesta:_


## 7. Test final y errores (15 % métricas; 20 % análisis/inferencia)
Cierra decisiones antes de consultar test. La tabla usa el test seleccionado a resolución de entrenamiento. La primera figura usa primeros casos: elige después dos éxitos/dos fallos (o mejores/peores) con criterio explícito y modelo de referencia.


In [ ]:
evaluation = ['--data', DATA, '--manifest', MANIFEST, '--fcn', RESULTS / 'fcn/best.pt',
              '--skip', RESULTS / 'skip/best.pt', '--out', RESULTS / 'comparison']
run('evaluate.py', *evaluation)
display(pd.read_csv(RESULTS / 'comparison/comparison.csv'))
cases = pd.read_csv(RESULTS / 'comparison/per_image.csv')
display(cases.sort_values('skip_iou_pet').head(8))
display(cases.sort_values('skip_iou_pet', ascending=False).head(8))


In [ ]:
EXAMPLES = [0, 1, 2, 3]  # Sustituye según criterio razonado
run('evaluate.py', *evaluation, '--examples', *EXAMPLES)
display(DisplayImage(filename=str(RESULTS / 'comparison/examples.png')))


**Selección y errores:** IDs, éxito/fallo, comparación, causas plausibles, borde ignorado y prueba futura. No cambies el modelo después de observar test; documenta cualquier corrección de un fallo real.

_Respuesta:_


## 8. T6 · Demo obligatoria en Linux local CPU (10 %)
Completa callback Gradio con cuatro entradas/salidas; nombre libre, None→gr.Error. Pruebas completas antes de entregar. Descargar ambos checkpoints y resultados con la celda inferior. En Linux: `python app.py --checkpoint results/skip/best.pt --device cpu`. Si puerto ocupado: --port 17861. No necesitas Internet para cargar el modelo ni GPU personal.

Dos fotos externas con derechos, tres fondos y sticker con alfa PNG; entrada vacía, tamaño/orientación y píxeles predichos como mascota conservados. Varias capturas en evidencias/; no vídeo. En Colab muestra inferencia, pero la demo se ejecuta obligatoriamente en local.


In [ ]:
run('-m', 'pytest', '-q')


In [ ]:
from PIL import Image
from src.inference import PetCut
petcut = PetCut(RESULTS / 'skip/best.pt')
photo_path = DATA / 'oxford-iiit-pet/images' / (ds.ids[0] + '.jpg')
with Image.open(photo_path) as im:
    photo = im.convert('RGB')
mask, overlay, edited, pair = petcut(photo, 'Fondo gris')
display(mask, overlay, edited)


**Evidencias de demo local:** describe las dos fotos externas, derechos, efectos, entrada vacía y comprobaciones; adjunta capturas o enlaces relativos a evidencias/. No inventes IoU sin máscaras externas. La comparación anterior con Oxford no sustituye estas pruebas.

_Respuesta y capturas:_


In [ ]:
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
environment_file = PERSIST / ('environment-colab.txt' if COLAB else 'environment-local.txt')
environment_file.write_text(environment, encoding='utf-8')
print('Versiones guardadas:', environment_file)


## 9. Descargar resultados y entregar
Esta celda crea un ZIP de transferencia a Linux, no la entrega final: contiene results/, splits/ y versiones, excluyendo last.pt. Descarga además el notebook ejecutado desde Archivo→Descargar .ipynb. Guarda last.pt en Drive para poder continuar.

En local incorpora código, memoria/INFORME.md + PDF derivado (máximo 4 páginas más portada), capturas, ambos entornos y pruebas propias. Sigue estructura de ENUNCIADO.md e INFORME_PLANTILLA.md. Ejecuta desde kernel limpio en modo verificar antes de entregar, sin repetir entrenamientos. No incluyas dataset ni entorno virtual.


In [ ]:
transfer = PROJECT / 'PetCut_resultados.zip'
with zipfile.ZipFile(transfer, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in (RESULTS, PERSIST / 'splits'):
        for path in sorted(folder.rglob('*')):
            if path.is_file() and path.name != 'last.pt' and not path.name.endswith('.tmp'):
                archive.write(path, path.relative_to(PERSIST).as_posix())
    if environment_file.exists():
        archive.write(environment_file, environment_file.name)
print('Descomprime en tu proyecto Linux:', transfer)
if COLAB:
    from google.colab import files
    files.download(str(transfer))
